# PLANTEUR — demo

Load a patch and visualize the Sentinel-2 image and the semantic annotation.
Set `PLANTEUR_ROOT` to the dataset folder.

In [ ]:
import sys, json, ast
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
sys.path.append("code")
from dataloader import PLANTEUR_Dataset

PLANTEUR_ROOT = "/weka/dfive-default/piperw/pastis2_drom_pastis_format"  # <- set me

label_names = json.load(open("documentation/label_names.json"))
cmap_rows = [ast.literal_eval(l.strip().rstrip(",")) for l in open("documentation/colormap.txt") if l.strip()]
cmap = ListedColormap(cmap_rows)
n_classes = len(label_names)
print(n_classes, "classes:", list(label_names.values()))

In [ ]:
# Load the test split (S2). Use sats=['S2','S1'] for optical+radar, target='instance' for panoptic.
ds = PLANTEUR_Dataset(folder=PLANTEUR_ROOT, sats=["S2"], target="semantic", folds=[5])
print("patches in fold 5 (test):", len(ds))
(data, dates), target = ds[0]
s2 = data["S2"] if isinstance(data, dict) else data
print("S2 time series:", tuple(s2.shape), "| target:", tuple(target.shape))

In [ ]:
# S2 true-color (B04,B03,B02 = channels 2,1,0) at a mid time step, next to the semantic label.
t = s2.shape[0] // 2
rgb = np.asarray(s2[t][[2, 1, 0]]).transpose(1, 2, 0).astype(np.float32)
lo, hi = np.percentile(rgb, [2, 98]); rgb = np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(rgb); ax[0].set_title(f"Sentinel-2 (t={t})"); ax[0].axis("off")
im = ax[1].imshow(np.asarray(target), cmap=cmap, vmin=0, vmax=n_classes - 1)
ax[1].set_title("Semantic labels"); ax[1].axis("off")
cbar = fig.colorbar(im, ax=ax[1], ticks=range(n_classes)); cbar.ax.set_yticklabels(list(label_names.values()))
plt.tight_layout(); plt.show()